# 03 — Pre-processing

This notebook prepares a supervised learning problem:

> Predict whether a trending entry in 2025 will fall into the top quartile of likes-to-views ratio, using only metadata-style features.

To keep the evaluation realistic, the model is trained on 2024 and tested on 2025.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

BASE_DIR = Path(".")
df = pd.read_csv(BASE_DIR / "yt_project_outputs" / "analysis_row_level.csv", parse_dates=["snapshot_date", "publish_date"])
print(df.shape)

(148567, 34)


In [2]:
model_df = df[np.isfinite(df["likes_to_views_ratio"])].copy()
cap = model_df["likes_to_views_ratio"].quantile(0.999)
model_df["likes_to_views_ratio"] = model_df["likes_to_views_ratio"].clip(upper=cap)

train = model_df[model_df["year"] == 2024].copy()
test = model_df[model_df["year"] == 2025].copy()

threshold = train["likes_to_views_ratio"].quantile(0.75)
train["high_engagement"] = (train["likes_to_views_ratio"] >= threshold).astype(int)
test["high_engagement"] = (test["likes_to_views_ratio"] >= threshold).astype(int)

threshold, train["high_engagement"].mean().round(4), test["high_engagement"].mean().round(4)

(np.float64(0.056434898389208274), np.float64(0.25), np.float64(0.1985))

In [3]:
features = [
    "country",
    "language_clean",
    "publish_weekday",
    "publish_hour_utc",
    "publish_month",
    "title_length",
    "description_length",
    "uppercase_ratio",
    "exclamation_count",
    "question_count",
    "hashtag_count",
    "tag_count",
]

categorical_features = ["country", "language_clean", "publish_weekday"]
numeric_features = [c for c in features if c not in categorical_features]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
])

X_train = train[features]
y_train = train["high_engagement"]
X_test = test[features]
y_test = test["high_engagement"]

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print("Train matrix shape:", X_train_t.shape)
print("Test matrix shape:", X_test_t.shape)

Train matrix shape: (99986, 226)
Test matrix shape: (48549, 226)


In [4]:
# Quick sanity table
prep_summary = pd.DataFrame({
    "dataset": ["train_2024", "test_2025"],
    "rows": [len(train), len(test)],
    "positive_rate": [y_train.mean(), y_test.mean()],
})
prep_summary.round(4)

,dataset,rows,positive_rate
0,train_2024,99986,0.2500
1,test_2025,48549,0.1985


## Summary

The modeling dataset is now ready:

- 2024 provides the training sample
- 2025 provides the holdout test sample
- the target is a high-engagement trending entry
- the feature set focuses on metadata and publishing context